In [37]:
import sys; sys.path.append('../3rdparty/ElasticKnots/3rdparty/ElasticRods/python')
import sys; sys.path.append('../3rdparty/ElasticKnots/python')
import elastic_rods, elastic_knots
import numpy as np, matplotlib.pyplot as plt, time, io, os
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import eigsh
from scipy.linalg import eigh

from helpers import *
from parametric_curves import *
import py_newton_optimizer 

from linkage_vis import LinkageViewer as Viewer, CenterlineViewer
from tri_mesh_viewer import PointCloudViewer, PointCloudMesh

%load_ext autoreload
%autoreload 2

import parallelism
parallelism.set_max_num_tbb_threads(1)

from MEP import MEP

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [58]:
knot_name = '6_2/0002.obj'
file = '../data/L400-r0.2-UpTo9Crossings/' + knot_name
rod_radius = 0.2
material = elastic_rods.RodMaterial('ellipse', 2000, 1, [rod_radius, rod_radius])
centerline = read_nodes_from_file(file)  # supported formats: obj, txt
pr = define_periodic_rod(centerline[::], material)
rod_list = elastic_knots.PeriodicRodList([pr])
len(rod_list.getDoFs())

1601

In [59]:
view = Viewer(rod_list, width=1024, height=800)
view.show()


Renderer(camera=PerspectiveCamera(aspect=1.28, children=(PointLight(color='#999999', position=(0.0, 0.0, 5.0),…

In [ ]:
def callback(problem, iteration):
    if iteration % 5 == 0:
        view.update()
for i in range(10,100):
    print(f"iterration: {i}")
    optimizerOptions = py_newton_optimizer.NewtonOptimizerOptions()
    optimizerOptions.niter = 1000
    optimizerOptions.gradTol = 1e-6
    hessianShift = 1e-4 * compute_min_eigenval_straight_rod(pr)

    problemOptions = elastic_knots.ContactProblemOptions()
    problemOptions.contactStiffness = 1e+3
    problemOptions.dHat = 2*rod_radius * 0.1*i
    fixedVars = []   
    
    report = elastic_knots.compute_equilibrium(
        rod_list, problemOptions, optimizerOptions, 
        fixedVars=fixedVars,
        externalForces=np.zeros(rod_list.numDoF()),
        softConstraints=[],
        callback=callback,
        hessianShift=hessianShift
        )
    view.update()

iterration: 10
0	0.920947	0.242217	0.242217	1	1
1	0.913161	0.000687487	0.000687487	1	1
2	0.913161	2.52588e-07	2.52588e-07	1	1
3	0.913161	1.59064e-09	1.59064e-09	1	0
4	0.913161	4.29372e-10	4.29372e-10	1	0
iterration: 11
0	10.3921	209.028	209.028	1	1
1	2.58331	63.7981	63.7981	1	1
2	1.2176	19.6744	19.6744	1	1
3	0.974671	5.87639	5.87639	1	1
4	0.93193	1.75808	1.75808	1	1
5	0.922253	0.562712	0.562712	1	1
6	0.918655	0.216067	0.216067	1	1
7	0.916853	0.0961878	0.0961878	1	1
8	0.915926	0.0458294	0.0458294	1	1
9	0.915449	0.0233846	0.0233846	1	1
10	0.915167	0.0131212	0.0131212	1	1
11	0.914976	0.00788597	0.00788597	1	1
12	0.914839	0.00470403	0.00470403	1	1
13	0.91474	0.00319789	0.00319789	1	1
14	0.914666	0.00192481	0.00192481	1	1
15	0.914609	0.00141878	0.00141878	1	1
16	0.914565	0.000994733	0.000994733	1	1
17	0.91453	0.000880452	0.000880452	1	1
18	0.9145	0.000800109	0.000800109	1	1
19	0.914477	0.000699989	0.000699989	1	1
20	0.91446	0.000605347	0.000605347	1	1
21	0.914447	0.000545988	0.000545988	1	1

The simulations will continue, but the result might be non-physical due to spurious contact forces between neighboring edges.
Consider decreasing the cross-section radius or using a coarser polyline.
Increasing the minContactEdgeDist parameter would remove non-physical contact forces, too, but only at the expense of topology preservation guarantees.



0	31.3755	642.69	642.69	1	1
1	5.97282	187.938	187.938	1	1
2	1.81166	55.9827	55.9827	1	1
3	1.09342	16.5574	16.5574	1	1
4	0.959595	4.32987	4.32987	1	1
5	0.938544	1.16205	1.16205	1	1
6	0.934256	0.329174	0.329174	1	1
7	0.932949	0.0975336	0.0975336	1	1
8	0.932464	0.0310034	0.0310034	1	1
9	0.932266	0.0112507	0.0112507	1	1
10	0.932157	0.00550314	0.00550314	1	1
11	0.932088	0.00461577	0.00461577	1	1
12	0.932041	0.00446098	0.00446098	1	1
13	0.932006	0.00409071	0.00409071	1	1
14	0.931978	0.00380175	0.00380175	1	1
15	0.931956	0.00313501	0.00313501	1	1
16	0.931938	0.00209194	0.00209194	1	1
17	0.931925	0.00165413	0.00165413	1	1
18	0.931917	0.000956861	0.000956861	1	1
19	0.931912	0.000578803	0.000578803	1	1
20	0.931908	0.000838872	0.000838872	1	1
21	0.931905	0.000876389	0.000876389	1	1
22	0.931904	0.00185179	0.00185179	0.5	1
23	0.931903	0.00647721	0.00647721	0.25	1
24	0.931903	0.00996323	0.00996323	0.25	1
25	0.931901	0.00105471	0.00105471	1	1
26	0.9319	9.48519e-05	9.48519e-05	1	1
27	0.931899	0.003085

The simulations will continue, but the result might be non-physical due to spurious contact forces between neighboring edges.
Consider decreasing the cross-section radius or using a coarser polyline.
Increasing the minContactEdgeDist parameter would remove non-physical contact forces, too, but only at the expense of topology preservation guarantees.



0	30.2792	640.012	640.012	1	1
1	5.04036	167.104	167.104	1	1
2	1.56054	45.3381	45.3381	1	1
3	1.03482	12.1098	12.1098	1	1
4	0.955029	3.21879	3.21879	1	1
5	0.940591	0.879664	0.879664	1	1
6	0.93659	0.267341	0.267341	1	1
7	0.93485	0.0882385	0.0882385	1	1
8	0.934064	0.0332578	0.0332578	1	1
9	0.933709	0.0146156	0.0146156	1	1
10	0.93353	0.00724832	0.00724832	1	1
11	0.933434	0.00497697	0.00497697	1	1
12	0.933372	0.00427286	0.00427286	1	1
13	0.933326	0.00387377	0.00387377	1	1
14	0.933291	0.00363399	0.00363399	1	1
15	0.933264	0.00295689	0.00295689	1	1
16	0.933244	0.00227661	0.00227661	1	1
17	0.933227	0.0019001	0.0019001	1	1
18	0.933214	0.0011603	0.0011603	1	1
19	0.933207	0.000655827	0.000655827	0.0625	0
20	0.933206	0.00666363	0.00666363	0.0625	1
21	0.933206	0.00623995	0.00623995	1	1
22	0.933199	0.00133089	0.00133089	1	1
23	0.933196	0.00030214	0.00030214	1	1
24	0.933193	0.000437756	0.000437756	1	1
25	0.933191	0.00104604	0.00104604	1	1
26	0.933191	0.0134598	0.0134598	0.25	1
27	0.93319	0.00727001	0.

...

0	1661.51	1735.34	1735.34	1	1
1	1241.08	7784.25	7784.25	1	1
2	1203.83	426.884	426.884	1	1
3	1160.73	427.049	427.049	1	1
4	1107.65	414.753	414.753	1	1
5	1051.62	388.659	388.659	1	1
6	1006.13	623.391	623.391	1	1
7	984.971	19168.7	19168.7	1	1
8	979.083	615.374	615.374	1	1
9	978.768	50.1919	50.1919	1	1
10	978.18	48.8631	48.8631	1	1
11	977.11	46.9667	46.9667	1	1
12	975.305	50.0172	50.0172	1	1
13	972.644	90.2936	90.2936	1	1
14	969.426	182.263	182.263	1	1
15	966.37	212.699	212.699	1	1
16	966.269	21.2397	21.2397	1	1
17	966.204	23.8603	23.8603	1	1
18	965.919	40.0376	40.0376	0.0625	1
19	965.151	374.061	374.061	0.5	1
20	964.035	154.492	154.492	1	1
21	963.241	243.547	243.547	1	1
22	962.461	721.42	721.42	0.5	1
23	961.507	718.22	718.22	1	1
24	960.324	357.329	357.329	1	1
25	959.334	860.948	860.948	1	1
26	958.986	179.231	179.231	1	1
27	958.515	68.6265	68.6265	1	1
28	957.888	85.5149	85.5149	1	1
29	957.074	90.1874	90.1874	1	1
30	955.994	92.0418	92.0418	1	1
31	954.561	135.657	135.657	1	1
32	952.831	329.9

The simulations will continue, but the result might be non-physical due to spurious contact forces between neighboring edges.
Consider decreasing the cross-section radius or using a coarser polyline.
Increasing the minContactEdgeDist parameter would remove non-physical contact forces, too, but only at the expense of topology preservation guarantees.



0	1916.64	1801.58	1801.58	1	1
1	1515.46	8455.52	8455.52	1	1
2	1444.78	39318.7	39318.7	1	1
3	1419.8	1330.58	1330.58	1	1
4	1415.05	326.1	326.1	1	1
5	1406.42	311.076	311.076	1	1
6	1391.28	292.746	292.746	1	1
7	1366.35	281.636	281.636	1	1
8	1329.48	315.334	315.334	1	1
9	1283.83	488.687	488.687	1	1
10	1240.11	477.909	477.909	1	1
11	1209.57	3088.42	3088.42	1	1
12	1206.84	109.351	109.351	1	1
13	1202.82	236.82	236.82	1	1
14	1197.76	398.813	398.813	1	1
15	1192.82	247.52	247.52	1	1
16	1189.11	1155.01	1155.01	1	1
17	1188.77	22.6677	22.6677	1	1
18	1188.25	89.0119	89.0119	1	1
19	1187.5	164.975	164.975	1	1
20	1186.58	211.912	211.912	1	1
21	1185.51	201.398	201.398	1	1
22	1184.3	200.92	200.92	1	1
23	1182.84	160.976	160.976	1	1
24	1181.09	163.728	163.728	1	1
25	1179.22	651.322	651.322	1	1
26	1179.19	3.06992	3.06992	1	1
27	1179.17	3.38203	3.38203	1	1
28	1179.12	3.46029	3.46029	1	1
29	1179.03	6.23134	6.23134	1	1
30	1178.86	14.6068	14.6068	1	1
31	1178.56	28.5047	28.5047	1	1
32	1178.1	40.978	40.978	1	1
33	

The simulations will continue, but the result might be non-physical due to spurious contact forces between neighboring edges.
Consider decreasing the cross-section radius or using a coarser polyline.
Increasing the minContactEdgeDist parameter would remove non-physical contact forces, too, but only at the expense of topology preservation guarantees.



0	2202.1	1871.46	1871.46	1	1
1	1817.32	9037.28	9037.28	1	1
2	1725.96	11667.2	11667.2	0.5	1
3	1718.55	22392.3	22392.3	1	1
4	1703.43	622.864	622.864	1	1
5	1695.36	349.488	349.488	1	1
6	1680.84	334.117	334.117	1	1
7	1656.18	338.061	338.061	1	1
8	1618.09	420.369	420.369	1	1
9	1567.73	503.499	503.499	1	1
10	1514.98	733.563	733.563	1	1
11	1474.01	1740.31	1740.31	1	1
12	1463.49	8092.69	8092.69	1	1
13	1462.47	68.6978	68.6978	1	1
14	1461.88	67.1853	67.1853	1	1
15	1460.76	65.2452	65.2452	1	1
16	1458.78	65.4517	65.4517	1	1
17	1455.57	93.4654	93.4654	1	1
18	1451.15	198.763	198.763	1	1
19	1446.3	247.966	247.966	1	1
20	1442.23	278.541	278.541	1	1
21	1439.42	1218.05	1218.05	0.5	1
22	1439.05	2195.68	2195.68	1	1
23	1438.97	12.1043	12.1043	1	1
24	1438.91	10.1975	10.1975	1	1
25	1438.81	10.5425	10.5425	1	1
26	1438.64	15.2005	15.2005	1	1
27	1438.33	34.9702	34.9702	1	1
28	1437.86	76.8507	76.8507	1	1
29	1437.18	122.86	122.86	1	1
30	1436.27	139.785	139.785	1	1
31	1435.07	142.736	142.736	1	1
32	1433.53	141.354

In [ ]:
from helpers import write_obj
file = '../data/NoCollision/' + knot_name
write_obj(file, rod_list)

In [ ]:
# Load the centerline from file...
file = '../data/NoCollision/' + knot_name
knot = read_nodes_from_file(file)
rod_radius = 0.2
material = elastic_rods.RodMaterial('ellipse', 2000, 0.3, [rod_radius, rod_radius])
pr = define_periodic_rod(knot[::4], material)
rod_list = elastic_knots.PeriodicRodList([pr])

In [ ]:
view = Viewer(rod_list, width=1024, height=800)
view.show()

In [ ]:
from helpers import write_obj
file = '../data/NoCollision/reduced' + knot_name
write_obj(file, rod_list)